In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc

# Parameter generation


In [7]:
60/39.6

1.5151515151515151

In [10]:
df_sampled

,k_off_gene_1,k_on_gene_1,mrna_half_life_gene_1,protein_half_life_gene_1,k_prod_protein_gene_1,burst_size_gene_1,k_off_gene_2,k_on_gene_2,mrna_half_life_gene_2,protein_half_life_gene_2,...,k_off_gene_3,k_on_gene_3,mrna_half_life_gene_3,protein_half_life_gene_3,k_prod_protein_gene_3,burst_size_gene_3,n_gene_1_to_gene_2,k_add_scaled_gene_1_to_gene_2,n_gene_2_to_gene_1,k_add_scaled_gene_2_to_gene_1
0,167.439330,0.222425,2.754900,18.281936,59.807297,24.247232,390.045863,0.366545,1.543892,27.229078,...,3.664188,0.396853,2.617529,16.486287,54.101553,36.441808,0.283664,4.283003,3.675379,7.872862
1,131.096118,0.352351,3.226193,29.818922,25.417880,35.405425,3.162048,0.424194,2.008094,28.070589,...,91.705647,0.176112,1.861013,24.563371,31.862596,33.875817,0.179652,0.794136,1.772654,19.617752
2,5.404775,0.273229,3.429285,40.324896,113.401770,25.226881,1.868475,0.427064,3.498178,18.261277,...,110.106257,0.382519,1.536762,43.897394,51.450961,35.307228,3.243156,5.593918,0.394667,4.406530
3,330.955413,0.288865,2.696422,25.056395,151.230774,42.226185,2.056647,0.150949,2.077876,55.046330,...,27.088531,0.194344,2.243158,17.298339,211.610855,24.120874,2.066992,17.441312,1.723406,2.073244
4,117.271558,0.194846,1.942725,30.389212,118.087588,22.801559,54.509682,0.398306,2.023971,47.683784,...,1.568209,0.188026,2.840664,24.551197,237.514049,48.177360,0.964510,0.643439,0.112053,2.926660
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,25.041817,0.383018,2.053052,46.684159,44.403912,51.173435,1.508647,0.143775,1.502699,38.329974,...,91.216679,0.145123,1.627369,15.451432,49.606360,25.337886,1.035229,15.850041,1.050943,14.699138
9996,1.751668,0.281132,2.771495,18.908470,75.314607,37.195500,12.733305,0.371570,1.522377,36.105311,...,210.081628,0.195110,1.535506,60.238831,42.654773,30.131331,0.735560,3.164979,0.189800,5.814501
9997,3.016802,0.439338,1.788875,27.294547,92.357980,37.557789,47.747404,0.212683,2.785371,52.264406,...,326.143415,0.351809,2.696613,16.550993,58.317477,38.361409,1.192841,4.262445,0.821572,1.328303
9998,317.461674,0.172502,3.271107,28.829046,43.426116,23.098272,62.253074,0.309129,1.732181,23.028537,...,20.612706,0.185505,1.975158,50.274023,173.049383,24.485560,1.020056,0.873466,0.236890,1.096797


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc

# --- Define sampled parameters ---
gene_params_sampled = [
    "k_off",            # k_on / (k_on + k_off)
    "k_on",             # directly sampled k_on
    "mrna_half_life",
    "protein_half_life",
    "k_prod_protein",
    "burst_size"
]

interaction_params_scaled = [
    "n_gene_1_to_gene_2", "k_add_scaled_gene_1_to_gene_2",
    "n_gene_2_to_gene_1", "k_add_scaled_gene_2_to_gene_1",
]

param_names = (
    [f"{p}_gene_1" for p in gene_params_sampled] +
    [f"{p}_gene_2" for p in gene_params_sampled] +
    [f"{p}_gene_3" for p in gene_params_sampled] +
    interaction_params_scaled
)

# --- Bounds for sampled parameters ---
param_bounds = {
    # Gene-level
    "k_off": (1.5, 480),                  # activation probability
    "k_on": (0.14, 0.47),                      # directly sampled
    "mrna_half_life": (1.4, 3.6),
    "protein_half_life": (15, 62),
    "burst_size": (20, 69),
    "k_prod_protein": (22, 309),
    # Interaction-level
    "n_gene_1_to_gene_2": (0.1, 5),
    "n_gene_2_to_gene_1": (0.1, 5),
    "k_add_scaled_gene_1_to_gene_2": (0.5, 20),
    "k_add_scaled_gene_2_to_gene_1": (0.5, 20),
}

bounds = (
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in interaction_params_scaled]
)

# --- Sampling configuration ---
n_valid_required = 10000
seed = 42

# Latin Hypercube Sampling (log10 for all except pi_on)
log_bounds_lower = [
    np.log10(b[0]) if "pi_on" not in param_names[i] else b[0]
    for i, b in enumerate(bounds)
]
log_bounds_upper = [
    np.log10(b[1]) if "pi_on" not in param_names[i] else b[1]
    for i, b in enumerate(bounds)
]

sampler = qmc.LatinHypercube(d=len(bounds), seed=seed)
sample = sampler.random(n=n_valid_required)

scaled_sample = np.empty_like(sample)
for i, name in enumerate(param_names):
    # Determine log bounds for every parameter (including pi_on)
    lower = np.log10(param_bounds["pi_on"][0]) if "pi_on" in name else log_bounds_lower[i]
    upper = np.log10(param_bounds["pi_on"][1]) if "pi_on" in name else log_bounds_upper[i]

    # Scale in log-space and exponentiate back
    scaled_log = qmc.scale(sample[:, [i]], [lower], [upper]).ravel()
    scaled_sample[:, i] = 10 ** scaled_log



df_sampled = pd.DataFrame(scaled_sample, columns=param_names)

# --- Convert to actual k_off and k_add ---
def convert_params(row):
    converted = {}

    for g in [1, 2, 3]:
        k_on = row[f"k_on_gene_{g}"]

        # Derive k_off from pi_on and k_on
        k_off = row[f"k_off_gene_{g}"]
        k_prod_mRNA = row[f'k_off_gene_{g}']*row[f'burst_size_gene_{g}']
        converted[f"k_on_gene_{g}"] = k_on
        converted[f"k_off_gene_{g}"] = k_off
        converted[f"mrna_half_life_gene_{g}"] = row[f"mrna_half_life_gene_{g}"]
        converted[f"protein_half_life_gene_{g}"] = row[f"protein_half_life_gene_{g}"]
        converted[f"k_prod_mRNA_gene_{g}"] = k_prod_mRNA
        converted[f"k_prod_protein_gene_{g}"] = row[f"k_prod_protein_gene_{g}"]
        converted[f"k_prod_mRNA_gene_{g}"] = k_prod_mRNA

    # Convert k_add_scaled → k_add
    for key in row.index:
        if "k_add_scaled" in key:
            tgt_gene = key.split("_")[7]  # target gene index
            k_on_target = converted[f"k_on_gene_{tgt_gene}"]
            converted[key.replace("k_add_scaled", "k_add")] = row[key] * k_on_target

        elif "n_gene" in key:
            converted[key] = row[key]

    return pd.Series(converted)

df_converted = df_sampled.apply(convert_params, axis=1)

# --- Expand to long format ---
rows = []
for idx, row in df_converted.iterrows():
    g1 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_1") and "to" not in k}
    g2 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_2") and "to" not in k}
    interactions = {k: v for k, v in row.items() if "gene_" in k and "to" in k}

    rows.append({**g1, **interactions, "pair_id": idx, "gene_id": 1})
    rows.append({**g2, **interactions, "pair_id": idx, "gene_id": 2})

final_df = pd.DataFrame(rows).reset_index(drop=True)

# --- Save ---
output_path = (
    "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_2genes_positive_reg_r_add_scaled_AG.csv"
)
final_df.to_csv(output_path)
print(f"\n✅ Saved to {output_path}")


✅ Saved to /home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_2genes_positive_reg_r_add_scaled_AG.csv


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc

# --- Define sampled parameters ---
gene_params_sampled = [
    "pi_on",            # k_on / (k_on + k_off)
    "k_on",             # directly sampled k_on
    "mrna_half_life",
    "protein_half_life",
    "k_prod_protein",
    "k_prod_mRNA"
]

interaction_params_scaled = [
    "n_gene_1_to_gene_2", "k_add_scaled_gene_1_to_gene_2",
    "n_gene_2_to_gene_1", "k_add_scaled_gene_2_to_gene_1",
    "n_gene_1_to_gene_3", "k_add_scaled_gene_1_to_gene_3",
    "n_gene_2_to_gene_3", "k_add_scaled_gene_2_to_gene_3"
]

param_names = (
    [f"{p}_gene_1" for p in gene_params_sampled] +
    [f"{p}_gene_2" for p in gene_params_sampled] +
    [f"{p}_gene_3" for p in gene_params_sampled] +
    interaction_params_scaled
)

# --- Bounds for sampled parameters ---
param_bounds = {
    # Gene-level
    "pi_on": (0.002, 0.4),                  # activation probability
    "k_on": (0.01, 3),                      # directly sampled
    "mrna_half_life": (0.6, 17),
    "protein_half_life": (7, 200),
    "k_prod_mRNA": (0.2, 60),
    "k_prod_protein": (19, 2700),
    # Interaction-level
    "n_gene_1_to_gene_2": (0.1, 5),
    "n_gene_2_to_gene_1": (0.1, 5),
    "n_gene_1_to_gene_3": (0.1, 5),
    "n_gene_2_to_gene_3": (0.1, 5),
    "k_add_scaled_gene_1_to_gene_2": (0.5, 2),
    "k_add_scaled_gene_2_to_gene_1": (0.5, 2),
    "k_add_scaled_gene_1_to_gene_3": (0.5, 2),
    "k_add_scaled_gene_2_to_gene_3": (0.5, 2),
}

bounds = (
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in gene_params_sampled] +
    [param_bounds[p] for p in interaction_params_scaled]
)

# --- Sampling configuration ---
n_valid_required = 25000
seed = 42

# Latin Hypercube Sampling (log10 for all except pi_on)
log_bounds_lower = [
    np.log10(b[0]) if "pi_on" not in param_names[i] else b[0]
    for i, b in enumerate(bounds)
]
log_bounds_upper = [
    np.log10(b[1]) if "pi_on" not in param_names[i] else b[1]
    for i, b in enumerate(bounds)
]

sampler = qmc.LatinHypercube(d=len(bounds), seed=seed)
sample = sampler.random(n=n_valid_required)

scaled_sample = np.empty_like(sample)
for i, name in enumerate(param_names):
    # Determine log bounds for every parameter (including pi_on)
    lower = np.log10(param_bounds["pi_on"][0]) if "pi_on" in name else log_bounds_lower[i]
    upper = np.log10(param_bounds["pi_on"][1]) if "pi_on" in name else log_bounds_upper[i]

    # Scale in log-space and exponentiate back
    scaled_log = qmc.scale(sample[:, [i]], [lower], [upper]).ravel()
    scaled_sample[:, i] = 10 ** scaled_log



df_sampled = pd.DataFrame(scaled_sample, columns=param_names)

# --- Convert to actual k_off and k_add ---
def convert_params(row):
    converted = {}

    for g in [1, 2, 3]:
        pi = row[f"pi_on_gene_{g}"]
        k_on = row[f"k_on_gene_{g}"]

        # Derive k_off from pi_on and k_on
        k_off = k_on * (1 - pi) / pi

        converted[f"k_on_gene_{g}"] = k_on
        converted[f"k_off_gene_{g}"] = k_off
        converted[f"mrna_half_life_gene_{g}"] = row[f"mrna_half_life_gene_{g}"]
        converted[f"protein_half_life_gene_{g}"] = row[f"protein_half_life_gene_{g}"]
        converted[f"k_prod_mRNA_gene_{g}"] = row[f"k_prod_mRNA_gene_{g}"]
        converted[f"k_prod_protein_gene_{g}"] = row[f"k_prod_protein_gene_{g}"]

    # Convert k_add_scaled → k_add
    for key in row.index:
        if "k_add_scaled" in key:
            tgt_gene = key.split("_")[7]  # target gene index
            k_on_target = converted[f"k_on_gene_{tgt_gene}"]
            converted[key.replace("k_add_scaled", "k_add")] = row[key] * k_on_target
        elif "n_gene" in key:
            converted[key] = row[key]

    return pd.Series(converted)

df_converted = df_sampled.apply(convert_params, axis=1)

# --- Expand to long format ---
rows = []
for idx, row in df_converted.iterrows():
    g1 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_1") and "to" not in k}
    g2 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_2") and "to" not in k}
    g3 = {k[:-7]: v for k, v in row.items() if k.endswith("_gene_3") and "to" not in k}
    interactions = {k: v for k, v in row.items() if "gene_" in k and "to" in k}

    rows.append({**g1, **interactions, "pair_id": idx, "gene_id": 1})
    rows.append({**g2, **interactions, "pair_id": idx, "gene_id": 2})
    rows.append({**g3, **interactions, "pair_id": idx, "gene_id": 3})

final_df = pd.DataFrame(rows).reset_index(drop=True)

# --- Save ---
output_path = (
    "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_repression_reg_pi_on_r_add_scaled.csv"
)
final_df.to_csv(output_path)
print(f"\n✅ Saved to {output_path}")


✅ Saved to /home/gzu5140/Keerthana_b1042/grnInference/simulation_data/parameter_scan_simulations/simulation_details/parameters_3genes_repression_reg_pi_on_r_add_scaled.csv


In [1]:
# param_df = pd.read_csv("/home/mzo5929/Keerthana/grnInference/simulation_data/gillespie_simulation/sim_details/lhc_sampled_parameters_negative_reg.csv", index_col = 0)

import numpy as np
import pandas as pd

def hl_to_deg(hl):
    """Convert half-life to degradation rate."""
    return np.log(2) / hl

def compute_steady_state_levels(param_df, gene_id):
    """Compute mean mRNA and protein levels for gene_id (1 or 2), assuming hill = 0.5."""
    assert gene_id in [1, 2], "gene_id must be 1 or 2"

    # Basic parameters
    k_on = param_df["k_on"]
    k_off = param_df["k_off"]
    prod_m = param_df["k_prod_mRNA"]
    prod_p = param_df["k_prod_protein"]
    deg_m = hl_to_deg(param_df["mrna_half_life"])
    deg_p = hl_to_deg(param_df["protein_half_life"])

    # Use .get to safely retrieve interaction term or default to 0
    if gene_id == 2:
        k_add = param_df.get("k_add_gene_1_to_gene_2", 0.0)
    else:
        k_add = param_df.get("k_add_gene_2_to_gene_1", 0.0)

    # Compute effective k_on using hill response = 0.5
    k_on_eff = k_on + 0.5 * k_add
    burst_prob = k_on_eff / (k_on_eff + k_off)

    # Steady-state means
    mean_mRNA = burst_prob * prod_m / deg_m
    mean_protein = mean_mRNA * prod_p / deg_p

    # Store results in DataFrame
    param_df["mean_mRNA_level"] = mean_mRNA
    param_df["mean_protein_level"] = mean_protein

    return param_df

# Usage
param_df = pd.read_csv("/home/mzo5929/Keerthana/grnInference/simulation_data/gillespie_simulation_run_2/sim_details/lhc_sampled_parameters_positive_reg_2.csv", index_col = 0)
param_df = compute_steady_state_levels(param_df, gene_id=2)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Filter out zero or negative values (log scale can't handle them)
values = param_df['mean_protein_level']
values = values[values > 0]

# Define log-spaced bins
n_bins = 100
min_val = values.min()
max_val = values.max()
log_bins = np.logspace(np.log10(min_val), np.log10(max_val), n_bins)

# Plot
plt.figure(figsize=(6, 4))
plt.hist(values, bins=log_bins)
plt.xscale('log')
# plt.yscale('log')
plt.xlabel("Mean protein level (log scale)")
plt.ylabel("Frequency (log scale)")
plt.title("Log-Binned Histogram of Mean Protein Levels")
plt.tight_layout()
plt.show()


In [ ]:
param_df[param_df['mean_mRNA_level'] < 100].shape

In [ ]:
plt.hist(param_df[param_df['mean_mRNA_level'] < 1]['mean_mRNA_level'])